# Baseline projector exports: 3 TSV modes

Questo notebook crea i **3 export TSV** per TensorFlow Projector:

1. **Aggregated trial representation**  
   1 punto = 1 trial aggregato  
   shape flatten: `(n_trials, 59*40)`

2. **Time-concatenated trial representation**  
   1 punto = 1 trial con le 5 window concatenate  
   shape flatten: `(n_trials, 5*59*40)`

3. **Window-level representation**  
   1 punto = 1 window  
   shape flatten: `(n_trials*5, 59*40)`

Data: 2026-03-06


In [7]:
import numpy as np
import pandas as pd
import torch
import json
from pathlib import Path


## Setup


In [8]:
project_root = Path("/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech")

SUBJECT_ID = 10

time_path = project_root / "data/processed/subject_tensors/subject_tensors_time" / f"subject_{SUBJECT_ID:02d}.pt"
agg_path  = project_root / "data/processed/subject_tensors/subject_tensors_aggregated_epoch" / f"subject_{SUBJECT_ID:02d}.pt"

assert time_path.exists(), f"Missing {time_path}"
assert agg_path.exists(), f"Missing {agg_path}"

label2idx_path = project_root / "data/interim/label2idx.json"
assert label2idx_path.exists(), f"Missing {label2idx_path}"

with open(label2idx_path, "r", encoding="utf-8") as f:
    label2idx = json.load(f)

idx2label = {int(v): k for k, v in label2idx.items()}


## Load tensors


In [9]:
obj_time = torch.load(time_path, map_location="cpu")
obj_agg  = torch.load(agg_path, map_location="cpu")

X_time = obj_time["X"].numpy()   # (n_trials, n_windows, n_channels, n_features)
y_time = obj_time["y"].numpy()
subject_time = obj_time["subject_id"].numpy()
session_time = obj_time["session_id"].numpy()
epoch_time = obj_time["epoch_idx"].numpy()

X_agg = obj_agg["X"].numpy()     # (n_trials, n_channels, n_features)
y_agg = obj_agg["y"].numpy()
subject_agg = obj_agg["subject_id"].numpy()
session_agg = obj_agg["session_id"].numpy()
epoch_agg = obj_agg["epoch_idx"].numpy()

print("X_time:", X_time.shape)
print("X_agg :", X_agg.shape)

assert X_time.shape[0] == X_agg.shape[0], "time vs agg trials mismatch"
assert np.all(y_time == y_agg), "time vs agg labels mismatch"


X_time: (550, 5, 59, 40)
X_agg : (550, 59, 40)


## Helpers


In [10]:
def words_from_y(y, idx2label):
    return [idx2label.get(int(lbl), f"UNK_{int(lbl)}") for lbl in y]

def save_projector_export(vectors, metadata_df, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    vectors_path = out_dir / "vectors.tsv"
    metadata_path = out_dir / "metadata.tsv"

    np.savetxt(vectors_path, vectors, delimiter="\t", fmt="%.6f")
    metadata_df.to_csv(metadata_path, sep="\t", index=False)

    print("Saved:")
    print(" ", vectors_path)
    print(" ", metadata_path)
    print(" vectors shape:", vectors.shape)
    print(" metadata shape:", metadata_df.shape)


## 1) Aggregated trial representation


In [11]:
# 1 punto = 1 trial aggregato
# X_agg: (n_trials, 59, 40) -> flatten (n_trials, 2360)

n_trials_agg = X_agg.shape[0]
X_agg_flat = X_agg.reshape(n_trials_agg, -1)
word_agg = words_from_y(y_agg, idx2label)

metadata_agg = pd.DataFrame({
    "sample_id": np.arange(n_trials_agg),
    "word": word_agg,
    "label_id": y_agg,
    "subject_id": subject_agg,
    "session_id": session_agg,
    "epoch_idx": epoch_agg,
    "representation": ["aggregated"] * n_trials_agg,
})

out_dir_agg = project_root / "data/processed/projector" / f"subject_{SUBJECT_ID:02d}_aggregated"
save_projector_export(X_agg_flat, metadata_agg, out_dir_agg)


Saved:
  /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/processed/projector/subject_10_aggregated/vectors.tsv
  /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/processed/projector/subject_10_aggregated/metadata.tsv
 vectors shape: (550, 2360)
 metadata shape: (550, 7)


## 2) Time-concatenated trial representation


In [12]:
# 1 punto = 1 trial
# X_time: (n_trials, 5, 59, 40) -> flatten (n_trials, 11800)

n_trials_time = X_time.shape[0]
X_time_concat = X_time.reshape(n_trials_time, -1)
word_time = words_from_y(y_time, idx2label)

metadata_time = pd.DataFrame({
    "sample_id": np.arange(n_trials_time),
    "word": word_time,
    "label_id": y_time,
    "subject_id": subject_time,
    "session_id": session_time,
    "epoch_idx": epoch_time,
    "representation": ["time_concatenated"] * n_trials_time,
})

out_dir_time = project_root / "data/processed/projector" / f"subject_{SUBJECT_ID:02d}_time_concatenated"
save_projector_export(X_time_concat, metadata_time, out_dir_time)


Saved:
  /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/processed/projector/subject_10_time_concatenated/vectors.tsv
  /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/processed/projector/subject_10_time_concatenated/metadata.tsv
 vectors shape: (550, 11800)
 metadata shape: (550, 7)


## 3) Window-level representation


In [13]:
# 1 punto = 1 window
# X_time: (n_trials, 5, 59, 40) -> (n_trials*5, 59*40)

n_trials, n_windows, n_channels, n_features = X_time.shape
X_window = X_time.reshape(n_trials * n_windows, n_channels * n_features)

word_window = []
label_window = []
subject_window = []
session_window = []
epoch_window = []
window_idx = []

for i in range(n_trials):
    for w in range(n_windows):
        word_window.append(idx2label.get(int(y_time[i]), f"UNK_{int(y_time[i])}"))
        label_window.append(int(y_time[i]))
        subject_window.append(int(subject_time[i]))
        session_window.append(int(session_time[i]))
        epoch_window.append(int(epoch_time[i]))
        window_idx.append(int(w))

metadata_window = pd.DataFrame({
    "sample_id": np.arange(n_trials * n_windows),
    "word": word_window,
    "label_id": label_window,
    "subject_id": subject_window,
    "session_id": session_window,
    "epoch_idx": epoch_window,
    "window": window_idx,
    "representation": ["window_level"] * (n_trials * n_windows),
})

out_dir_window = project_root / "data/processed/projector" / f"subject_{SUBJECT_ID:02d}_window_level"
save_projector_export(X_window, metadata_window, out_dir_window)


Saved:
  /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/processed/projector/subject_10_window_level/vectors.tsv
  /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/processed/projector/subject_10_window_level/metadata.tsv
 vectors shape: (2750, 2360)
 metadata shape: (2750, 8)


## Quick recap


In [14]:
print("Done. Created 3 projector exports for subject", SUBJECT_ID)
print()
print("1) Aggregated       -> 1 point = 1 trial, time collapsed")
print("2) Time-concat      -> 1 point = 1 trial, 5 windows concatenated")
print("3) Window-level     -> 1 point = 1 window")


Done. Created 3 projector exports for subject 10

1) Aggregated       -> 1 point = 1 trial, time collapsed
2) Time-concat      -> 1 point = 1 trial, 5 windows concatenated
3) Window-level     -> 1 point = 1 window
